# Task 1 — Problem Definition & Dataset
**Module:** 7006SCN Machine Learning and Big Data

---

## 1.1 Problem Definition

I decided this problem should be to predict whether a US domestic flight will arrive late. I defined a late flight as one which had an arrival delay greater than or equal to 15 minutes. This is actually how the FAA defines a late flight, and it's the metric the US government also uses. Thus, label 1 means the flight arrived at least 15 minutes late and label 0 means the flight arrived on time or early.

The problem choice was largely made on the grounds of real-world relevance. Airlines want to minimize delays as they increase operational costs and lose revenue. Airports also have an interest in better planning for gate usage. Passengers might even be interested if they can be assured a flight will not be late if they are on a tight connection. Having a predictive model that only needs scheduled information about the flight is therefore valuable.

Features in the model include those that would be known at the time the flight is booked: airline carrier, origin and destination airports, day of week, month of the year, scheduled flight duration, and the scheduled time of departure. I made the decision to avoid using the actual delay of departure, since that would involve leakage, as knowledge of the departure delay would not be available until the flight has already taken off.

---

## 1.2 Dataset

| Property | Detail |
|----------|--------|
| **Dataset** | US Airline On-Time Performance |
| **Source** | Bureau of Transportation Statistics (BTS) |
| **URL** | https://transtats.bts.gov |
| **Format** | ZIP-compressed CSV (one file per month) |
| **Rows** | ~13.3 million flights (2022–2023) |
| **Columns** | 110 (we use 10 for modelling) |

The dataset was downloaded directly from the BTS PREZIP server — 24 monthly files covering January 2022 through December 2023, totalling 6,130 MB of raw CSV data. This is a genuine operational reporting dataset produced by a US Federal agency from real flight records, not a pre-packaged Kaggle extract.

---

## 1.3 Why This Is Big Data — The 5 V's

In week 1, we talked about the 5 V's framework. Here's how the airline dataset fits in:

**Volume** — Every commercial domestic US flight since 1987 is tracked by the BTS. There are more than 180 million entries across the set. Even just from the last 2 years (2022 and 2023) yields a dataset of around 13.3 million rows which goes well over the minimum of 10 million required. Even at this scale, you cannot handle the data processing with the typical library of pandas. A normal laptop with your setup will not support processing the entire dataset without throwing the notorious "out of memory error".

**Velocity** — Ideally, you'll want flight data coming in in real time as it originates from flight booking platforms, airport reporting systems, etc. Any departure or arrival will be reported to the systems as soon as it takes place, whether actual or scheduled. Although, BTS data is released in monthly chunks as opposed to a real time feed for each event. The underlying source domain is still a high-velocity operational reporting pipeline — what we use here is a historical snapshot of it.

**Variety** — We have a plethora of variable types in our dataset. Some columns are numerical (e.g. delay minutes, distance, elapsed time), others are categorical (e.g. airline codes, airport codes, state names), we also have time based features (scheduled departure/arrival, actual departure/arrival) and then there are standard flags (cancelled, diverted, delayed). Each of them require a unique pre-processing approach.

**Veracity** — The dataset comes with some caveats regarding data quality. Cancelled flights result in many NaN values when it comes to their actual times of arrival or their delay minutes; diverted flights can lead to quite bizarre delays which will not be represented well in standard delay models; we can find some NaN entries within the tail number field; and finally, we get cases where the calculated flight delay is negative (a flight arriving prior to schedule), as well as numerous entries with far outlier values. Some columns (e.g. CRSElapsedTime) also contain implausible negative values that reflect reporting errors rather than true flight data. All that needs to be sorted out beforehand.

**Value** — The ability to predict flight delays has a significant financial upside. As mentioned already, flight delays are estimated to cost the US billions of dollars annually according to FAA estimates. A predictive modelling tool will be valuable not only for passengers trying to plan out trips but also for airlines and airports.

---

## 1.4 Ethics and Licensing

This is the U.S. Federal Agency BTS On-Time Performance dataset, and is free and Public Domain. This dataset is freely usable for academic use under the terms of the U.S. Open Government Data Policy.

It contains no passenger name, flight number, flight details or any other personal identifying information of any kind. It is purely a collection of airline performance statistics for delays by route.

The one bias within this data is if it is used for some sort of fairness analysis of airlines themselves. Many smaller and some of the larger and some commuter regional airlines, because of the history of old and aging equipment, have much more difficulty meeting on-time performance. There is no discrimination by any airlines within this data itself — but because a person lives somewhere and gets to a small hub, there would be much longer times. The models here really are just based on historical operations.

---

## Cell 1 — Environment Setup

Same Java 17 setup as the labs — has to point to the spark311 conda env or PySpark won't start.

In [1]:
# set JAVA_HOME before importing pyspark - has to be done first
import os, sys, time, pyspark
from pathlib import Path

os.environ['JAVA_HOME'] = '/opt/anaconda3/envs/spark311'
os.environ['PATH'] = '/opt/anaconda3/envs/spark311/bin:' + os.environ.get('PATH', '')

print('Python:', sys.version)
print('JAVA_HOME:', os.environ['JAVA_HOME'])

Python: 3.11.15 (main, Jun 11 2026, 15:14:57) [Clang 20.1.8 ]
JAVA_HOME: /opt/anaconda3/envs/spark311


## Cell 2 — SparkSession

Same config as the lab — `local[2]`, 8g driver memory, 20 shuffle partitions. Keeping it consistent with the lab setup.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType

spark = (
    SparkSession.builder
    .master('local[2]')
    .appName('7006SCN_Airline_Task1')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '20')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

DATA_DIR = Path('/tmp/airline')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Spark version:', spark.version)
print('Master:', spark.sparkContext.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 20:52:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Master: local[2]


## Cell 3 — Download Data

Downloading monthly ZIP files from the BTS PREZIP server for 2022 and 2023 — that's 24 files, each around 25-30 MB compressed. Much faster than a paginated API since each file is a direct download. Using curl with retry logic in case any individual file times out.

In [3]:
# download monthly airline on-time data from BTS
# 24 ZIP files (2022 + 2023), each ~25 MB - much faster than a paginated API
import subprocess, zipfile, time
from pathlib import Path

DATA_DIR = Path('/tmp/airline')
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE  = 'https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present'
YEARS = [2022, 2023]

downloaded = 0
skipped    = 0
failed     = []

for year in YEARS:
    for month in range(1, 13):
        csv_path = DATA_DIR / f'airline_{year}_{month:02d}.csv'
        if csv_path.exists() and csv_path.stat().st_size > 100_000:
            skipped += 1
            continue

        url     = f'{BASE}_{year}_{month}.zip'
        zip_tmp = f'/tmp/airline/_tmp_{year}_{month}.zip'

        # try up to 3 times with increasing wait
        ok = False
        for attempt in range(3):
            cmd = ['curl', '-s', '-L', '--retry', '2', '--retry-delay', '5',
                   '--max-time', '90', '-o', zip_tmp, url]
            result = subprocess.run(cmd, capture_output=True)
            zip_path = Path(zip_tmp)
            if result.returncode == 0 and zip_path.exists() and zip_path.stat().st_size > 10_000:
                ok = True
                break
            wait = 15 * (attempt + 1)
            print(f'  retry {attempt+1} for {year}-{month:02d}, waiting {wait}s...', flush=True)
            time.sleep(wait)

        if not ok:
            print(f'  FAILED: {year}-{month:02d}')
            failed.append(f'{year}-{month:02d}')
            Path(zip_tmp).unlink(missing_ok=True)
            continue

        # extract the CSV from inside the ZIP
        try:
            with zipfile.ZipFile(zip_tmp) as zf:
                csv_names = [n for n in zf.namelist() if n.endswith('.csv')]
                if csv_names:
                    zf.extract(csv_names[0], path=str(DATA_DIR))
                    extracted = DATA_DIR / csv_names[0]
                    extracted.rename(csv_path)
        except Exception as e:
            print(f'  ZIP error {year}-{month:02d}: {e}')
            failed.append(f'{year}-{month:02d}')
            continue
        finally:
            Path(zip_tmp).unlink(missing_ok=True)

        mb = csv_path.stat().st_size / 1e6
        print(f'  {year}-{month:02d}: {mb:.0f} MB', flush=True)
        downloaded += 1
        time.sleep(3)   # pause between downloads to avoid rate limiting

print(f'\nDone. Downloaded: {downloaded}, Skipped: {skipped}')
if failed:
    print(f'Failed months: {failed}')
total_mb = sum(f.stat().st_size for f in DATA_DIR.glob('airline_*.csv')) / 1e6
print(f'Total CSV size: {total_mb:.0f} MB')

  2022-01: 241 MB
  2022-02: 223 MB
  2022-03: 255 MB
  2022-04: 251 MB
  2022-05: 261 MB
  2022-06: 260 MB
  2022-07: 269 MB
  2022-08: 266 MB
  2022-09: 251 MB
  2022-10: 259 MB
  2022-11: 247 MB
  2022-12: 251 MB
  2023-01: 243 MB
  2023-02: 227 MB
  2023-03: 263 MB
  2023-04: 254 MB
  2023-05: 262 MB
  2023-06: 261 MB
  2023-07: 273 MB
  2023-08: 273 MB
  2023-09: 257 MB
  2023-10: 271 MB
  2023-11: 255 MB
  2023-12: 258 MB

Done. Downloaded: 24, Skipped: 0
Total CSV size: 6130 MB


## Cell 4 — File Size Verification

**Figure 1.1 — File size verification.** There are 24 CSV monthly datasets, none of these are null and in total 6,130 MB of raw data, thus it does surpass the scale required for Spark distributed processing and is not capable of being processed by a pandas workflow for a single-machine setup.

In [4]:
# check all 24 monthly files are present and non-empty
print('=' * 55)
print('FILE SIZE VERIFICATION')
print('=' * 55)
total = 0
files = sorted(DATA_DIR.glob('airline_*.csv'))
for f in files:
    mb = f.stat().st_size / 1e6
    total += mb
    print(f'  {f.name:<35} {mb:>6.0f} MB')
print('-' * 55)
print(f'  Files : {len(files)} / 24 expected')
print(f'  TOTAL : {total:.0f} MB')
print('=' * 55)

FILE SIZE VERIFICATION
  airline_2022_01.csv                    241 MB
  airline_2022_02.csv                    223 MB
  airline_2022_03.csv                    255 MB
  airline_2022_04.csv                    251 MB
  airline_2022_05.csv                    261 MB
  airline_2022_06.csv                    260 MB
  airline_2022_07.csv                    269 MB
  airline_2022_08.csv                    266 MB
  airline_2022_09.csv                    251 MB
  airline_2022_10.csv                    259 MB
  airline_2022_11.csv                    247 MB
  airline_2022_12.csv                    251 MB
  airline_2023_01.csv                    243 MB
  airline_2023_02.csv                    227 MB
  airline_2023_03.csv                    263 MB
  airline_2023_04.csv                    254 MB
  airline_2023_05.csv                    262 MB
  airline_2023_06.csv                    261 MB
  airline_2023_07.csv                    273 MB
  airline_2023_08.csv                    273 MB
  airline_2023_09

## Cell 5 — Load Into Spark

Reading all 24 monthly CSVs into a single Spark DataFrame in one go. The BTS files all have the same schema so the glob load works cleanly. Using DISK_ONLY persistence since 12M rows won't fit comfortably in memory.

In [5]:
import pyspark
from pyspark.storagelevel import StorageLevel

t0 = time.time()

# read all monthly CSVs in one shot - they all have the same schema
df = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(str(DATA_DIR / 'airline_*.csv'))
)

df = df.persist(StorageLevel.DISK_ONLY)
row_count = df.count()

print(f'Loaded in {time.time()-t0:.1f}s')
print(f'Rows      : {row_count:,}')
print(f'Columns   : {len(df.columns)}')
print(f'Partitions: {df.rdd.getNumPartitions()}')

Loaded in 76.0s
Rows      : 13,577,024
Columns   : 110
Partitions: 48


## Cell 6 — Schema: `df.printSchema()`

**Figure 1.2 — Schema confirmation.** `df.printSchema()` confirms that we have a mixed type dataset: some integer type features (Year, Month, DayOfWeek), some float type features (Distance, CRSElapsedTime, ArrDel15, ArrDelay), some string type features (Reporting_Airline, Origin, Dest, OriginState) and some date type features (FlightDate). This means the dataset is suitable for mixed type feature engineering using StringIndexer for the string type features, and StandardScaler for the numeric features.

In [6]:
# print the schema
print('DATASET SCHEMA')
print('=' * 50)
df.printSchema()

DATASET SCHEMA
root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- DOT_ID_Reporting_Airline: integer (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: integer (nullable = true)
 |-- OriginAirportID: integer (nullable = true)
 |-- OriginAirportSeqID: integer (nullable = true)
 |-- OriginCityMarketID: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateFips: integer (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- OriginWac: integer (nullable = true)
 |-- DestAirportID: integer (nullable = true)
 |-- Des

## Cell 7 — First 20 Rows: `df.show(20)`

**Figure 1.3 — Row sample.** `df.show(20)` verifies the form of a multi-feature flight-level table containing the categorical (Reporting_Airline, Origin, Dest), numeric (CRSDepTime, CRSElapsedTime, Distance), and binary target (ArrDel15) fields that will be used in supervised classification tasks. In the sample, the 19th row (ArrDel15 = NULL, Cancelled = 1.0) represents one of the reasons why cancelled flights should not be included when building a model, since there is no delay outcome available.

In [7]:
# look at the first 20 rows of the raw data
# selecting the key columns we care about - the full 110 columns would be too wide
df.select(
    'Year', 'Month', 'DayOfWeek', 'Reporting_Airline',
    'Origin', 'Dest', 'CRSDepTime', 'CRSElapsedTime',
    'Distance', 'ArrDel15', 'Cancelled'
).show(20)

+----+-----+---------+-----------------+------+----+----------+--------------+--------+--------+---------+
|Year|Month|DayOfWeek|Reporting_Airline|Origin|Dest|CRSDepTime|CRSElapsedTime|Distance|ArrDel15|Cancelled|
+----+-----+---------+-----------------+------+----+----------+--------------+--------+--------+---------+
|2022|    1|        5|               YX|   CMH| DCA|      1224|          88.0|   323.0|     0.0|      0.0|
|2022|    1|        6|               YX|   CMH| DCA|      1224|          88.0|   323.0|     0.0|      0.0|
|2022|    1|        7|               YX|   CMH| DCA|      1224|          88.0|   323.0|     0.0|      0.0|
|2022|    1|        1|               YX|   CMH| DCA|      1224|          88.0|   323.0|     0.0|      0.0|
|2022|    1|        2|               YX|   CMH| DCA|      1224|          88.0|   323.0|     0.0|      0.0|
|2022|    1|        3|               YX|   CMH| DCA|      1224|          88.0|   323.0|     0.0|      0.0|
|2022|    1|        4|               

## Cell 8 — Exploratory Analysis

Checking the overall delay rate and distribution across airlines before creating the label.

In [8]:
# explore the key distributions
print('Delay rate by airline:')
df.filter(F.col('Cancelled') == 0).groupBy('Reporting_Airline') \
  .agg(
      F.count('*').alias('flights'),
      F.round(F.mean('ArrDel15') * 100, 1).alias('pct_delayed')
  ).orderBy('pct_delayed', ascending=False).show(15)

print('Flights by month:')
df.groupBy('Month').count().orderBy('Month').show(12)

print('Distance and elapsed time stats:')
df.select('Distance', 'CRSElapsedTime', 'ArrDelay').describe().show()

Delay rate by airline:
+-----------------+-------+-----------+
|Reporting_Airline|flights|pct_delayed|
+-----------------+-------+-----------+
|               B6| 531928|       31.9|
|               F9| 324795|       31.6|
|               G4| 227524|       30.0|
|               NK| 484812|       27.1|
|               HA| 153099|       23.6|
|               WN|2688634|       22.9|
|               AA|1778804|       22.8|
|               UA|1336529|       20.0|
|               AS| 465533|       20.0|
|               YV| 110969|       19.7|
|               OH| 399411|       18.0|
|               MQ| 467706|       17.7|
|               QX|  87279|       16.6|
|               OO|1387156|       16.5|
|               YX| 588774|       16.4|
+-----------------+-------+-----------+
only showing top 15 rows
Flights by month:
+-----+-------+
|Month|  count|
+-----+-------+
|    1|1076739|
|    2| 998462|
|    3|1145175|
|    4|1117943|
|    5|1158777|
|    6|1154545|
|    7|1196823|
|    8|1192797

## Cell 9 — Create Binary Label

The `ArrDel15` column is already the label I need — it's 1 if the arrival delay was 15 minutes or more, 0 otherwise. This is the FAA's official definition of a delayed flight. I just need to filter out cancelled and diverted flights (which have null delay values) and cast to double for the ML pipeline.

In [9]:
# create the label - ArrDel15 is already 1/0 from BTS
# drop cancelled and diverted flights - they have null delay values
# also drop rows where ArrDel15 itself is null

df_labeled = (
    df
    .filter(F.col('Cancelled') == 0)
    .filter(F.col('Diverted') == 0)
    .filter(F.col('ArrDel15').isNotNull())
    .filter(F.col('CRSElapsedTime').isNotNull())
    .filter(F.col('Distance').isNotNull())
    .withColumn('label', F.col('ArrDel15').cast(DoubleType()))
)

df_labeled = df_labeled.persist(StorageLevel.DISK_ONLY)
labeled_count = df_labeled.count()
pos = df_labeled.filter(F.col('label') == 1.0).count()
neg = labeled_count - pos

print(f'Total rows after filter : {labeled_count:,}')
print(f'Delayed (label=1)       : {pos:,} ({100*pos/labeled_count:.1f}%)')
print(f'On time (label=0)       : {neg:,} ({100*neg/labeled_count:.1f}%)')
print()
print(f'Rows >= 10M  : {labeled_count >= 10_000_000}')
print(f'Cols >= 10   : {len(df.columns) >= 10}')

Total rows after filter : 13,275,415
Delayed (label=1)       : 2,763,497 (20.8%)
On time (label=0)       : 10,511,918 (79.2%)

Rows >= 10M  : True
Cols >= 10   : True


In [10]:
# save as parquet so Task 2 can load without repeating all this
LABELED_DIR = str(DATA_DIR / 'labeled')
df_labeled.write.mode('overwrite').parquet(LABELED_DIR)

print(f'Saved to: {LABELED_DIR}')
print(f'Rows saved: {labeled_count:,}')
print()
print('Task 1 complete — run Task2.ipynb next')

Saved to: /tmp/airline/labeled
Rows saved: 13,275,415

Task 1 complete — run Task2.ipynb next
